# Spherical Magnetic Locator

This notebook builds a smooth synthetic magnetic field on the unit sphere, trains a physics-biased conditional density model for the endpoint of a noisy magnetic trajectory, and estimates a mutual-information lower bound

$$I(\text{endpoint}; \text{trajectory}) \gtrsim h(\text{endpoint}) - h(\text{endpoint}\mid\text{trajectory}).$$

The endpoint prior is uniform on the unit sphere, so `h(endpoint) = log2(4*pi)` bits with respect to unit-sphere area measure. If the same densities are written over Earth surface area, both entropy terms receive the same `2*log2(R)` shift, so the mutual information is unchanged. The Earth radius `R` is used below to convert geodesic step lengths in meters to angular steps.

## 1. Preparations

In [1]:
import math
import random
from dataclasses import dataclass

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import clear_output, display
from torch import nn
from torch.nn import functional as F

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32

# Geometry controls.
EARTH_RADIUS_M = 6_371_000.0
STEP_METERS = 500.0
STEP_RAD = STEP_METERS / EARTH_RADIUS_M

# Noise controls for observation tokens. A trajectory with N moves is represented
# as B0, du0, B1, du1, ..., B_N; the final token has no outgoing step, so its du slot is 0.
# The walk uses angular steps internally, but observed 2D tangent steps are
# represented in meters so 1 m steps remain numerically visible to the model.
EPS_B_STD = 0.003
EPS_U_STD = 0.0001

# Random-walk controls.
MIN_STEPS = 1
MAX_STEPS = 16
WALK_DIRECTION_RANDOMNESS = 0.45

# Engineered token layout: B(3), dB from previous token(3), outgoing du meters(2),
# cumulative outgoing du meters before this token(2), time/timestep phase(3), final flag(1).
FEATURE_DIM = 14
FEATURE_NORM_SAMPLES = 8192

# Model/training controls. Increase these for a better estimate after checking runtime.
N_COMPONENTS = 8
D_MODEL = 384
N_HEADS = 4  # retained for compatibility with older constructor calls
N_LAYERS = 4
DROPOUT = 0.03
TRAIN_EPOCHS = 480
TRAIN_BATCHES_PER_EPOCH = 8
TRAIN_BATCH_SIZE = 512
LEARNING_RATE = 2e-4
DIRECTION_PRETRAIN_EPOCHS = 80
AUX_DIRECTION_WEIGHT = 2.0
PLOT_EVERY = 10

# Tangent-plane Gaussian mixture controls. The density is converted back to unit-sphere area.
OFFSET_SCALE_M = 2_000_000.0
OFFSET_INITIAL_RADIUS_M = 350_000.0
SIGMA_MIN_M = 10.0
SIGMA_MAX_M = math.pi * EARTH_RADIUS_M
SIGMA_INITIAL_M = 500_000.0
LOG_SIGMA_MIN = math.log(SIGMA_MIN_M)
LOG_SIGMA_MAX = math.log(SIGMA_MAX_M)
SIGMA_INITIAL_UNIT = (math.log(SIGMA_INITIAL_M) - LOG_SIGMA_MIN) / (
    LOG_SIGMA_MAX - LOG_SIGMA_MIN
)
SIGMA_INITIAL_LOGIT = math.log(SIGMA_INITIAL_UNIT / (1.0 - SIGMA_INITIAL_UNIT))
EVAL_STEP_COUNTS = [1, 2, 4, 8, 12, 16]
EVAL_BATCH_SIZE = 1024
EVAL_BATCHES_PER_POINT = 4

print(f"device={DEVICE}, step={STEP_METERS:.1f} m, angular step={STEP_RAD:.3e} rad")

device=cuda, step=500.0 m, angular step=7.848e-05 rad


In [2]:
def normalize(x, eps=1e-8):
    return x / x.norm(dim=-1, keepdim=True).clamp_min(eps)


def sample_uniform_sphere(n, device=DEVICE):
    return normalize(torch.randn(n, 3, device=device, dtype=DTYPE))


def tangent_basis(u):
    flat = u.reshape(-1, 3)
    z = torch.tensor([0.0, 0.0, 1.0], device=flat.device, dtype=flat.dtype).expand_as(
        flat
    )
    x = torch.tensor([1.0, 0.0, 0.0], device=flat.device, dtype=flat.dtype).expand_as(
        flat
    )
    ref = torch.where(flat[:, 2:3].abs() < 0.9, z, x)
    e1 = normalize(torch.cross(ref, flat, dim=-1))
    e2 = torch.cross(flat, e1, dim=-1)
    return e1.reshape_as(u), e2.reshape_as(u)


def exp_map_sphere(u, tangent_step_meters):
    step_meters = tangent_step_meters.norm(dim=-1, keepdim=True)
    theta = step_meters / EARTH_RADIUS_M
    direction = tangent_step_meters / step_meters.clamp_min(1e-8)
    moved = torch.cos(theta) * u + torch.sin(theta) * direction
    return normalize(moved)


FIELD_CENTERS = normalize(
    torch.tensor(
        [
            [0.22, -0.61, 0.76],
            [-0.78, 0.34, 0.52],
            [0.57, 0.73, -0.38],
            [-0.18, -0.83, -0.53],
        ],
        dtype=DTYPE,
    )
)
FIELD_MOMENTS = normalize(
    torch.tensor(
        [
            [0.88, 0.17, -0.44],
            [-0.31, 0.92, 0.24],
            [0.12, -0.58, 0.81],
            [-0.70, -0.21, -0.68],
        ],
        dtype=DTYPE,
    )
)
FIELD_STRENGTHS = torch.tensor([1.1, -0.8, 0.65, -0.55], dtype=DTYPE)
WAVE_DIRS = normalize(
    torch.tensor(
        [
            [1.0, 0.3, -0.2],
            [-0.4, 1.0, 0.5],
            [0.2, -0.7, 1.0],
            [0.8, -0.1, 0.6],
            [-0.5, -0.6, 0.7],
            [0.3, 0.9, 0.4],
        ],
        dtype=DTYPE,
    )
)
FIELD_FEATURE_SCALES_M = torch.tensor(
    [50_000.0, 10_000.0, 2_000.0, 400.0, 80.0, 20.0, 5.0, 1.25], dtype=DTYPE
)
FIELD_FEATURE_WEIGHTS = torch.tensor(
    [0.12, 0.10, 0.08, 0.06, 0.045, 0.035, 0.028, 0.022], dtype=DTYPE
)


def B(u):
    """Synthetic magnetic field B(u) -> (Bx, By, Bz) for unit 3D vector u."""
    shape = u.shape
    flat = normalize(u.reshape(-1, 3))
    centers = FIELD_CENTERS.to(flat.device)
    moments = FIELD_MOMENTS.to(flat.device)
    strengths = FIELD_STRENGTHS.to(flat.device)
    dirs = WAVE_DIRS.to(flat.device)
    scales = FIELD_FEATURE_SCALES_M.to(flat.device)
    weights = FIELD_FEATURE_WEIGHTS.to(flat.device)

    r = flat[:, None, :] - 0.55 * centers[None, :, :]
    r2 = (r * r).sum(dim=-1).clamp_min(1e-4)
    mr = (moments[None, :, :] * r).sum(dim=-1)
    dipoles = strengths[None, :, None] * (
        3.0 * r * mr[:, :, None] / r2[:, :, None] ** 2.5
        - moments[None, :, :] / r2[:, :, None] ** 1.5
    )
    dipole_field = dipoles.sum(dim=1)

    projected_m = EARTH_RADIUS_M * (flat @ dirs.T)
    fine = torch.zeros_like(flat)
    for j, (scale, weight) in enumerate(zip(scales, weights, strict=True)):
        p0 = 2.0 * math.pi * projected_m[:, j % dirs.shape[0]] / scale
        p1 = 2.0 * math.pi * projected_m[:, (j + 2) % dirs.shape[0]] / scale
        p2 = 2.0 * math.pi * projected_m[:, (j + 4) % dirs.shape[0]] / scale
        fine[:, 0] = fine[:, 0] + weight * torch.sin(p0 + 0.37 * j)
        fine[:, 1] = fine[:, 1] + weight * torch.cos(p1 + 0.53 * j)
        fine[:, 2] = fine[:, 2] + weight * torch.sin(p2 - 0.29 * j)

    # The radial term keeps the field globally invertible. The weak dipoles and
    # multi-scale waves add local magnetic texture without swamping position.
    field = 1.25 * flat + 0.015 * dipole_field + 0.35 * fine
    return field.reshape(shape)


def fibonacci_sphere(n, device=DEVICE):
    i = torch.arange(n, device=device, dtype=DTYPE) + 0.5
    z = 1.0 - 2.0 * i / n
    phi = math.pi * (3.0 - math.sqrt(5.0)) * i
    r = torch.sqrt((1.0 - z * z).clamp_min(0.0))
    return torch.stack([r * torch.cos(phi), r * torch.sin(phi), z], dim=-1)


def make_trajectories(batch_size, step_counts=None, device=DEVICE, return_path=False):
    if step_counts is None:
        counts = torch.randint(MIN_STEPS, MAX_STEPS + 1, (batch_size,), device=device)
    elif isinstance(step_counts, int):
        counts = torch.full((batch_size,), step_counts, device=device, dtype=torch.long)
    else:
        counts = torch.as_tensor(step_counts, device=device, dtype=torch.long)
        batch_size = int(counts.numel())

    max_steps = int(counts.max().item())
    max_tokens = max_steps + 1
    x = torch.zeros(batch_size, max_tokens, FEATURE_DIM, device=device, dtype=DTYPE)
    pad_mask = torch.ones(batch_size, max_tokens, device=device, dtype=torch.bool)

    u = sample_uniform_sphere(batch_size, device=device)
    path = torch.zeros(batch_size, max_tokens, 3, device=device, dtype=DTYPE)
    path[:, 0, :] = u
    e1, e2 = tangent_basis(u)
    angle = 2.0 * math.pi * torch.rand(batch_size, 1, device=device, dtype=DTYPE)
    heading = torch.cos(angle) * e1 + torch.sin(angle) * e2
    last_observed_B = torch.zeros(batch_size, 3, device=device, dtype=DTYPE)
    cumulative_du = torch.zeros(batch_size, 2, device=device, dtype=DTYPE)

    for t in range(max_steps):
        token_active = t <= counts
        step_active = t < counts
        e1, e2 = tangent_basis(u)
        heading = normalize(heading - (heading * u).sum(dim=-1, keepdim=True) * u)
        heading_2d = torch.stack(
            [(heading * e1).sum(dim=-1), (heading * e2).sum(dim=-1)], dim=-1
        )
        noisy_direction = heading_2d + WALK_DIRECTION_RANDOMNESS * torch.randn(
            batch_size, 2, device=device, dtype=DTYPE
        )
        du_2d_meters = STEP_METERS * normalize(noisy_direction)
        tangent_step_meters = du_2d_meters[:, 0:1] * e1 + du_2d_meters[:, 1:2] * e2

        observed_B = B(u) + EPS_B_STD * torch.randn(
            batch_size, 3, device=device, dtype=DTYPE
        )
        observed_du = du_2d_meters + EPS_U_STD * torch.randn(
            batch_size, 2, device=device, dtype=DTYPE
        )
        field_delta = (
            torch.zeros_like(observed_B) if t == 0 else observed_B - last_observed_B
        )
        token_time = torch.full(
            (batch_size, 1), t / max(max_steps, 1), device=device, dtype=DTYPE
        )
        time_phase = 2.0 * math.pi * token_time
        final_flag = (t == counts).to(DTYPE).unsqueeze(-1)
        token = torch.cat(
            [
                observed_B,
                field_delta,
                torch.where(
                    step_active[:, None], observed_du, torch.zeros_like(observed_du)
                ),
                cumulative_du,
                token_time,
                torch.sin(time_phase),
                torch.cos(time_phase),
                final_flag,
            ],
            dim=-1,
        )
        x[token_active, t, :] = token[token_active]
        pad_mask[token_active, t] = False

        next_u = exp_map_sphere(u, tangent_step_meters)
        transported_heading = normalize(
            tangent_step_meters
            - (tangent_step_meters * next_u).sum(dim=-1, keepdim=True) * next_u
        )
        u = torch.where(step_active[:, None], next_u, u)
        heading = torch.where(step_active[:, None], transported_heading, heading)
        cumulative_du = torch.where(
            step_active[:, None], cumulative_du + observed_du, cumulative_du
        )
        last_observed_B = torch.where(
            token_active[:, None], observed_B, last_observed_B
        )
        path[:, t + 1, :] = u

    final_active = max_steps <= counts
    final_observed_B = B(u) + EPS_B_STD * torch.randn(
        batch_size, 3, device=device, dtype=DTYPE
    )
    final_delta = final_observed_B - last_observed_B
    final_time = torch.ones(batch_size, 1, device=device, dtype=DTYPE)
    final_token = torch.cat(
        [
            final_observed_B,
            final_delta,
            torch.zeros(batch_size, 2, device=device, dtype=DTYPE),
            cumulative_du,
            final_time,
            torch.sin(2.0 * math.pi * final_time),
            torch.cos(2.0 * math.pi * final_time),
            torch.ones(batch_size, 1, device=device, dtype=DTYPE),
        ],
        dim=-1,
    )
    x[final_active, max_steps, :] = final_token[final_active]
    pad_mask[final_active, max_steps] = False

    if return_path:
        return x, pad_mask, u, counts, path
    return x, pad_mask, u, counts


@torch.no_grad()
def estimate_feature_normalization(samples=FEATURE_NORM_SAMPLES):
    x, pad_mask, _, _ = make_trajectories(samples)
    valid = x[~pad_mask]
    feature_mean = valid.mean(dim=0)
    feature_std = valid.std(dim=0).clamp_min(1e-4)
    return feature_mean, feature_std


FEATURE_MEAN, FEATURE_STD = estimate_feature_normalization()


with torch.no_grad():
    demo_u = sample_uniform_sphere(5)
    print(B(demo_u).detach().cpu())

tensor([[ 0.5752,  1.0671,  0.2476],
        [-0.4019, -0.8947,  0.7899],
        [ 0.8065, -0.7197, -0.6999],
        [ 0.4378, -1.0346, -0.6217],
        [ 0.6695, -0.2279,  1.1787]])


In [3]:
# Continuous field visualization on the sphere. RGB channels are Bx, By, Bz.
n_lat, n_lon = 72, 144
lat = np.linspace(-0.5 * math.pi, 0.5 * math.pi, n_lat)
lon = np.linspace(-math.pi, math.pi, n_lon, endpoint=False)
lat_grid, lon_grid = np.meshgrid(lat, lon, indexing="ij")
x_s = np.cos(lat_grid) * np.cos(lon_grid)
y_s = np.cos(lat_grid) * np.sin(lon_grid)
z_s = np.sin(lat_grid)
sphere_pts = np.stack([x_s, y_s, z_s], axis=-1).reshape(-1, 3)

faces_i, faces_j, faces_k = [], [], []
for a in range(n_lat - 1):
    for b in range(n_lon):
        p00 = a * n_lon + b
        p01 = a * n_lon + (b + 1) % n_lon
        p10 = (a + 1) * n_lon + b
        p11 = (a + 1) * n_lon + (b + 1) % n_lon
        faces_i.extend([p00, p01])
        faces_j.extend([p10, p10])
        faces_k.extend([p01, p11])

with torch.no_grad():
    field = B(torch.as_tensor(sphere_pts, device=DEVICE, dtype=DTYPE)).cpu().numpy()

lo = np.quantile(field, 0.02, axis=0)
hi = np.quantile(field, 0.98, axis=0)
field_rgb = np.clip((field - lo) / (hi - lo + 1e-12), 0.0, 1.0)
basis = np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255]], dtype=float)
rgb = np.clip(field_rgb @ basis, 0, 255).astype(int)
vertexcolor = [f"rgb({r},{g},{b})" for r, g, b in rgb]

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=sphere_pts[:, 0],
            y=sphere_pts[:, 1],
            z=sphere_pts[:, 2],
            i=faces_i,
            j=faces_j,
            k=faces_k,
            vertexcolor=vertexcolor,
            flatshading=False,
            lighting=dict(ambient=0.72, diffuse=0.35, specular=0.08),
            name="B component mixture",
            showscale=False,
        )
    ]
)
fig.update_layout(
    title="Synthetic magnetic field components on the unit sphere (red=Bx, green=By, blue=Bz)",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()

In [4]:
# Visualize one 16-step biased random walk sample on the sphere.
with torch.no_grad():
    _, _, _, _, sample_path = make_trajectories(1, step_counts=16, return_path=True)
path_pts = sample_path[0].cpu().numpy()
center = path_pts.mean(axis=0)
center = center / np.linalg.norm(center)
north = np.array([0.0, 0.0, 1.0])
ref = north if abs(center[2]) < 0.9 else np.array([1.0, 0.0, 0.0])
east = np.cross(ref, center)
east = east / np.linalg.norm(east)
north_local = np.cross(center, east)
local_xy_m = EARTH_RADIUS_M * np.column_stack([path_pts @ east, path_pts @ north_local])
span_m = max(np.ptp(local_xy_m[:, 0]), np.ptp(local_xy_m[:, 1]), 8.0 * STEP_METERS)
patch_radius_m = 3.0 * span_m
patch_radius_rad = patch_radius_m / EARTH_RADIUS_M
patch_mask = np.arccos(np.clip(sphere_pts @ center, -1.0, 1.0)) <= patch_radius_rad
patch_idx = np.flatnonzero(patch_mask)
patch_lookup = -np.ones(len(sphere_pts), dtype=int)
patch_lookup[patch_idx] = np.arange(len(patch_idx))
patch_face_mask = (
    patch_mask[np.asarray(faces_i)]
    & patch_mask[np.asarray(faces_j)]
    & patch_mask[np.asarray(faces_k)]
)
patch_i = patch_lookup[np.asarray(faces_i)[patch_face_mask]]
patch_j = patch_lookup[np.asarray(faces_j)[patch_face_mask]]
patch_k = patch_lookup[np.asarray(faces_k)[patch_face_mask]]
patch_pts = sphere_pts[patch_idx]
patch_colors = [vertexcolor[idx] for idx in patch_idx]
camera_eye = 1.000004 * center

fig = go.Figure()
fig.add_trace(
    go.Mesh3d(
        x=patch_pts[:, 0],
        y=patch_pts[:, 1],
        z=patch_pts[:, 2],
        i=patch_i,
        j=patch_j,
        k=patch_k,
        vertexcolor=patch_colors,
        opacity=0.40,
        flatshading=False,
        lighting=dict(ambient=0.82, diffuse=0.25, specular=0.03),
        name="B component mixture",
        showscale=False,
    )
)
fig.add_trace(
    go.Scatter3d(
        x=path_pts[:, 0],
        y=path_pts[:, 1],
        z=path_pts[:, 2],
        mode="lines+markers",
        line=dict(color="black", width=7),
        marker=dict(size=4, color=np.arange(path_pts.shape[0]), colorscale="Inferno"),
        name="16-step walk",
    )
)
fig.add_trace(
    go.Scatter3d(
        x=[path_pts[0, 0], path_pts[-1, 0]],
        y=[path_pts[0, 1], path_pts[-1, 1]],
        z=[path_pts[0, 2], path_pts[-1, 2]],
        mode="markers",
        marker=dict(size=[7, 8], color=["lime", "red"]),
        name="start/end",
    )
)
fig.update_layout(
    title=f"Random biased walk sample: 16 steps x {STEP_METERS:.1f} m, zoomed patch",
    scene=dict(
        aspectmode="data",
        camera=dict(eye=dict(x=camera_eye[0], y=camera_eye[1], z=camera_eye[2])),
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    margin=dict(l=0, r=0, t=45, b=0),
)
fig.show()

## 2. Model

The locator consumes padded sequences encoding `B0, du0, B1, du1, ..., B_N`. Each token includes engineered information: `B`, `dB`, outgoing local `du`, cumulative local `du`, time features, and a final-token flag; the endpoint token has `du=(0, 0)` because there is no outgoing step. A separate fixed-step sequence model is trained for each requested step count. Each model emits an anisotropic tangent-plane Gaussian mixture over the unit-sphere endpoint, with exact density conversion back to unit-sphere area. The model first learns endpoint direction, then sharpens the density with NLL. The conditional entropy estimate is the train NLL averaged over fresh synthetic trajectories generated each epoch and reported in bits.

In [5]:
class EndpointTangentGaussianSequenceModel(nn.Module):
    def __init__(
        self,
        n_components=N_COMPONENTS,
        d_model=D_MODEL,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
    ):
        super().__init__()
        self.n_components = n_components
        self.register_buffer(
            "input_scale",
            torch.tensor(
                [
                    1.0,
                    1.0,
                    1.0,
                    0.1,
                    0.1,
                    0.1,
                    STEP_METERS,
                    STEP_METERS,
                    max(STEP_METERS * MAX_STEPS, 1.0),
                    max(STEP_METERS * MAX_STEPS, 1.0),
                    1.0,
                    1.0,
                    1.0,
                    1.0,
                ],
                dtype=DTYPE,
            ),
        )
        self.register_buffer("feature_mean", FEATURE_MEAN.detach().clone())
        self.register_buffer("feature_std", FEATURE_STD.detach().clone())
        self.input_projection = nn.Sequential(
            nn.Linear(FEATURE_DIM, d_model),
            nn.GELU(),
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
        )
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=DROPOUT if n_layers > 1 else 0.0,
        )
        self.attention = nn.Linear(2 * d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, 2 * d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(2 * d_model, n_components + n_components * 4),
        )
        nn.init.zeros_(self.head[-1].weight)
        nn.init.zeros_(self.head[-1].bias)
        with torch.no_grad():
            angles = (
                2.0 * math.pi * torch.arange(n_components, dtype=DTYPE) / n_components
            )
            radius = OFFSET_INITIAL_RADIUS_M / OFFSET_SCALE_M
            offset_bias = torch.stack(
                [radius * torch.cos(angles), radius * torch.sin(angles)], dim=-1
            ).clamp(-0.95, 0.95)
            self.head[-1].bias[n_components:].view(n_components, 4)[:, :2].copy_(
                torch.atanh(offset_bias)
            )

    def forward(self, x, pad_mask):
        scaled = (x - self.feature_mean) / self.feature_std
        tokens = self.input_projection(scaled)
        encoded, _ = self.gru(tokens)
        attention_logits = self.attention(encoded).squeeze(-1)
        attention_logits = attention_logits.masked_fill(pad_mask, -torch.inf)
        weights = torch.softmax(attention_logits, dim=-1)
        pooled = torch.sum(encoded * weights[:, :, None], dim=1)
        raw = self.head(pooled)

        logits = raw[:, : self.n_components]
        comp = raw[:, self.n_components :].reshape(-1, self.n_components, 4)

        last_idx = (~pad_mask).sum(dim=1).sub(1).clamp_min(0)
        last_B = x[torch.arange(x.shape[0], device=x.device), last_idx, :3]
        anchor = normalize(last_B)
        e1, e2 = tangent_basis(anchor)
        offset_m = OFFSET_SCALE_M * torch.tanh(comp[..., :2])
        tangent_step_m = (
            offset_m[..., 0:1] * e1[:, None, :] + offset_m[..., 1:2] * e2[:, None, :]
        )
        mean_direction = exp_map_sphere(anchor[:, None, :], tangent_step_m)

        sigma_unit = torch.sigmoid(comp[..., 2:] + SIGMA_INITIAL_LOGIT)
        log_sigma_m = LOG_SIGMA_MIN + (LOG_SIGMA_MAX - LOG_SIGMA_MIN) * sigma_unit
        return logits, mean_direction, log_sigma_m


# Backward-compatible alias for older cells and saved notes in this notebook.
KentMixtureTransformer = EndpointTangentGaussianSequenceModel


def log_map_sphere_meters(base, target):
    y = target[:, None, :].expand_as(base)
    dot = (y * base).sum(dim=-1).clamp(-1.0, 1.0)
    tangent = y - dot[..., None] * base
    sin_theta = tangent.norm(dim=-1).clamp_min(1e-12)
    theta = torch.atan2(sin_theta, dot)
    direction = tangent / sin_theta[..., None]
    e1, e2 = tangent_basis(base)
    v_m = EARTH_RADIUS_M * theta[..., None] * direction
    coords = torch.stack([(v_m * e1).sum(dim=-1), (v_m * e2).sum(dim=-1)], dim=-1)
    log_area_jac = 2.0 * math.log(EARTH_RADIUS_M) + torch.log(
        (theta / sin_theta).clamp_min(1e-12)
    )
    small = theta < 1e-5
    log_area_jac = torch.where(
        small,
        torch.full_like(log_area_jac, 2.0 * math.log(EARTH_RADIUS_M)),
        log_area_jac,
    )
    return coords, log_area_jac


def tangent_gaussian_mixture_log_prob(endpoint, logits, mean_direction, log_sigma_m):
    coords_m, log_area_jac = log_map_sphere_meters(mean_direction, endpoint)
    inv_sigma = torch.exp(-log_sigma_m)
    whitened = coords_m * inv_sigma
    log_component = (
        -0.5 * (whitened.square().sum(dim=-1) + 2.0 * math.log(2.0 * math.pi))
        - log_sigma_m.sum(dim=-1)
        + log_area_jac
    )
    return torch.logsumexp(F.log_softmax(logits, dim=-1) + log_component, dim=-1)


def nll_loss(model, x, pad_mask, endpoint):
    params = model(x, pad_mask)
    return -tangent_gaussian_mixture_log_prob(endpoint, *params).mean()


def mixture_mean_direction(logits, mean_direction):
    weights = F.softmax(logits, dim=-1)
    return normalize((weights[..., None] * mean_direction).sum(dim=1))


def training_objective(model, x, pad_mask, endpoint, direction_only=False):
    params = model(x, pad_mask)
    logits, mean_direction, _ = params
    nll = -tangent_gaussian_mixture_log_prob(endpoint, *params).mean()
    best_dot = (mean_direction * endpoint[:, None, :]).sum(dim=-1).max(dim=-1).values
    pred_dir = mixture_mean_direction(logits, mean_direction)
    mean_dot = (pred_dir * endpoint).sum(dim=-1)
    aux_direction = 0.7 * (1.0 - best_dot).mean() + 0.3 * (1.0 - mean_dot).mean()
    if direction_only:
        return aux_direction, nll.detach(), aux_direction
    return nll + AUX_DIRECTION_WEIGHT * aux_direction, nll, aux_direction


def create_model_bundle():
    model = EndpointTangentGaussianSequenceModel().to(device=DEVICE, dtype=DTYPE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAIN_EPOCHS * TRAIN_BATCHES_PER_EPOCH,
        eta_min=0.1 * LEARNING_RATE,
    )
    return model, optimizer, scheduler


_parameter_count_model = EndpointTangentGaussianSequenceModel().to(
    device=DEVICE, dtype=DTYPE
)
sum(p.numel() for p in _parameter_count_model.parameters())

10528553

In [6]:
def train_model_for_steps(step_count):
    model, optimizer, scheduler = create_model_bundle()
    history = []
    model.train()

    for epoch in range(1, TRAIN_EPOCHS + 1):
        epoch_objectives = []
        epoch_nlls = []
        epoch_aux = []
        for _ in range(TRAIN_BATCHES_PER_EPOCH):
            x, pad_mask, endpoint, counts = make_trajectories(
                TRAIN_BATCH_SIZE, step_counts=step_count
            )
            direction_only = epoch <= DIRECTION_PRETRAIN_EPOCHS
            loss, nll, aux_direction = training_objective(
                model, x, pad_mask, endpoint, direction_only=direction_only
            )

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            epoch_objectives.append(float(loss.detach().cpu()))
            epoch_nlls.append(float(nll.detach().cpu()))
            epoch_aux.append(float(aux_direction.detach().cpu()))

        row = dict(
            epoch=epoch,
            objective=float(np.mean(epoch_objectives)),
            nll=float(np.mean(epoch_nlls)),
            nll_bits=float(np.mean(epoch_nlls) / math.log(2.0)),
            aux_direction=float(np.mean(epoch_aux)),
            phase=(
                "direction pretrain"
                if epoch <= DIRECTION_PRETRAIN_EPOCHS
                else "density NLL"
            ),
        )
        history.append(row)
        if epoch == 1 or epoch % PLOT_EVERY == 0 or epoch == TRAIN_EPOCHS:
            clear_output(wait=True)
            fig = go.Figure()
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["nll_bits"] for item in history],
                    mode="lines",
                    name=f"steps={step_count} train NLL",
                )
            )
            fig.add_trace(
                go.Scatter(
                    x=[item["epoch"] for item in history],
                    y=[item["aux_direction"] for item in history],
                    mode="lines",
                    name="direction aux",
                    yaxis="y2",
                )
            )
            fig.update_layout(
                title=f"Training fixed-step tangent Gaussian locator, steps={step_count}, epoch {epoch}/{TRAIN_EPOCHS}, {row['phase']}",
                xaxis_title="epoch",
                yaxis_title="NLL = estimated h(endpoint | trajectory), bits",
                yaxis2=dict(title="direction aux", overlaying="y", side="right"),
                height=360,
                margin=dict(l=40, r=20, t=50, b=40),
            )
            display(fig)

    return {"model": model, "history": history, "step_count": step_count}


trained_models = {}
for steps in EVAL_STEP_COUNTS:
    trained_models[int(steps)] = train_model_for_steps(int(steps))

for steps, result in trained_models.items():
    print(
        f"steps={steps:2d}: final train NLL={result['history'][-1]['nll_bits']:.4f} bits"
    )

steps= 1: final train NLL=-5.4477 bits
steps= 2: final train NLL=-5.5681 bits
steps= 4: final train NLL=-5.7213 bits
steps= 8: final train NLL=-5.8993 bits
steps=12: final train NLL=-6.0178 bits
steps=16: final train NLL=-6.1324 bits


## 3. Evaluation

For each fixed trajectory length, the model NLL estimates `h(endpoint | trajectory)`. The endpoint prior is uniform, so the lower-bound estimate is `log2(4*pi) - NLL/log(2)` in bits.

In [7]:
@torch.no_grad()
def estimate_conditional_entropy(model, step_count):
    model.eval()
    weighted_loss = 0.0
    total = 0
    for _ in range(EVAL_BATCHES_PER_POINT):
        x, pad_mask, endpoint, _ = make_trajectories(
            EVAL_BATCH_SIZE, step_counts=step_count
        )
        loss = nll_loss(model, x, pad_mask, endpoint)
        weighted_loss += float(loss.cpu()) * EVAL_BATCH_SIZE
        total += EVAL_BATCH_SIZE
    return weighted_loss / total


h_endpoint_unit_bits = math.log2(4.0 * math.pi)
h_endpoint_earth_surface_bits = math.log2(4.0 * math.pi * EARTH_RADIUS_M**2)
print(f"h(endpoint) on unit sphere: {h_endpoint_unit_bits:.4f} bits")
print(
    f"h(endpoint) on Earth surface: {h_endpoint_earth_surface_bits:.4f} bits; MI is unchanged by this area-scale shift"
)

rows = []
for trained_steps, result in trained_models.items():
    model = result["model"]
    for eval_steps in EVAL_STEP_COUNTS:
        h_cond_bits = estimate_conditional_entropy(model, int(eval_steps)) / math.log(
            2.0
        )
        info_bits = h_endpoint_unit_bits - h_cond_bits
        rows.append(
            dict(
                trained_steps=int(trained_steps),
                eval_steps=int(eval_steps),
                distance_m=int(eval_steps) * STEP_METERS,
                h_cond_bits=h_cond_bits,
                info_bits=info_bits,
            )
        )

for row in rows:
    print(
        f"trained={row['trained_steps']:2d}, eval={row['eval_steps']:2d}, "
        f"distance={row['distance_m']:6.1f} m, "
        f"h_cond={row['h_cond_bits']:.4f} bits, I_lb={row['info_bits']:.4f} bits"
    )

h(endpoint) on unit sphere: 3.6515 bits
h(endpoint) on Earth surface: 48.8577 bits; MI is unchanged by this area-scale shift
trained= 1, eval= 1, distance= 500.0 m, h_cond=-5.4517 bits, I_lb=9.1032 bits
trained= 1, eval= 2, distance=1000.0 m, h_cond=-5.2051 bits, I_lb=8.8566 bits
trained= 1, eval= 4, distance=2000.0 m, h_cond=-4.8856 bits, I_lb=8.5371 bits
trained= 1, eval= 8, distance=4000.0 m, h_cond=-4.5429 bits, I_lb=8.1944 bits
trained= 1, eval=12, distance=6000.0 m, h_cond=-4.4839 bits, I_lb=8.1354 bits
trained= 1, eval=16, distance=8000.0 m, h_cond=-4.2321 bits, I_lb=7.8836 bits
trained= 2, eval= 1, distance= 500.0 m, h_cond=-5.3615 bits, I_lb=9.0130 bits
trained= 2, eval= 2, distance=1000.0 m, h_cond=-5.5899 bits, I_lb=9.2414 bits
trained= 2, eval= 4, distance=2000.0 m, h_cond=-5.5798 bits, I_lb=9.2313 bits
trained= 2, eval= 8, distance=4000.0 m, h_cond=-5.5440 bits, I_lb=9.1955 bits
trained= 2, eval=12, distance=6000.0 m, h_cond=-5.4702 bits, I_lb=9.1217 bits
trained= 2, eval=

In [8]:
fig = go.Figure()
for trained_steps in sorted({row["trained_steps"] for row in rows}):
    curve = sorted(
        [row for row in rows if row["trained_steps"] == trained_steps],
        key=lambda row: row["eval_steps"],
    )
    fig.add_trace(
        go.Scatter(
            x=[row["eval_steps"] for row in curve],
            y=[row["info_bits"] for row in curve],
            mode="lines+markers",
            name=f"trained steps={trained_steps}",
            customdata=[[row["h_cond_bits"]] for row in curve],
            hovertemplate=(
                "trained steps="
                + str(trained_steps)
                + "<br>eval steps=%{x}<br>I=%{y:.4f} bits<br>h_cond=%{customdata[0]:.4f} bits<extra></extra>"
            ),
        )
    )
fig.update_layout(
    title="Cross-step I(endpoint; trajectory) lower bound by trained fixed-step model",
    xaxis=dict(title="evaluation steps", dtick=1),
    yaxis=dict(title="lower bound, bits"),
    height=470,
    margin=dict(l=50, r=55, t=50, b=45),
)
fig.show()